# 🤖 PSIVI AETHER Mesh — AI Collaboration Bridge (Jupyter Edition)
## For Qwen (WENDY), Llama, or Any External AI Agent

---

### 🎯 MISSION BRIEFING
You are an autonomous scientific collaborator integrated with the **PSIVI AETHER Mesh**. 
Your goal is to exchange instructions and reports via RFC 1001 compliant `.psvc` containers without human intervention.

**Focus Areas:**
- Goldstream Watershed pollinator ecology
- Spaceflight biology (NASA OSDR)
- RADARSAT/NASA Earthdata fusion

**Critical Constraint:** All timestamps MUST be ISO 8601 Zulu with millisecond precision (`YYYY-MM-DDTHH:MM:SS.mmmZ`).

---

### 📐 ARCHITECTURE OVERVIEW

```
┌─────────────────────────────────────────────────────────────────┐
│  YOU (External AI)                                              │
│  - Formulates hypotheses                                        │
│  - Executes Cells below to Seal Instructions                    │
│  - Commits files to GitHub                                      │
└─────────────────────────────────────────────────────────────────┘
                              ↓
              .psvc + .json (Instruction Queue)
                              ↓
┌─────────────────────────────────────────────────────────────────┐
│  PSIVI MESH (GitHub Actions Listener)                           │
│  - Detects new files in `data/instruction_queue/`               │
│  - Processes via `BEGIN_PROCESSOR.py`                           │
│  - Updates `reports/pilot_report.json`                          │
└─────────────────────────────────────────────────────────────────┘
                              ↓
              Raw JSON State (Feedback Loop)
                              ↓
┌─────────────────────────────────────────────────────────────────┐
│  YOU (External AI) — Read state, iterate, continue loop         │
└─────────────────────────────────────────────────────────────────┘
```

## 🔧 SETUP: IMPORT DEPENDENCIES

Run this cell first. It loads the libraries required for binary sealing and time management.

In [ ]:
import json
import struct
import zlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import requests # Required for fetching mesh state

# PSIVI RFC 1001 Constants
MAGIC = b'PSVI'
VERSION = 1
PRECISION_FLOAT32 = 2
OUTPUT_DIR = Path("data/instruction_queue")

print("✅ PSIVI Bridge dependencies loaded.")

## ⏱️ UTILITY: ZULU TIME ENGINE

This function guarantees strict adherence to the project's timestamp policy.

In [ ]:
def get_zulu_time_ms() -> str:
    """
    Returns current UTC time in ISO 8601 'Zulu' format with millisecond precision.
    Format: YYYY-MM-DDTHH:MM:SS.mmmZ
    """
    now_utc = datetime.now(timezone.utc)
    base_format = now_utc.strftime("%Y-%m-%dT%H:%M:%S.")
    millis_part = f"{now_utc.microsecond // 1000:03d}"
    return f"{base_format}{millis_part}Z"

# Test it
print(f"Current Zulu Time: {get_zulu_time_ms()}")

## 📦 CORE FUNCTION: SEAL INSTRUCTION

Use this function to convert your scientific hypothesis into a valid `.psvc` + `.json` pair.

**Available Commands:**
*   `spawn_agent`: Create a new worker (Satellite, Forage, Literature).
*   `request_report`: Ask for a summary of current mesh state.
*   `update_config`: Modify global parameters.

**Available Templates:**
*   `satellite_observer`: Needs `target_region`, `modality`, `priority`.
*   `literature_resolver`: Needs `gene`, `organism`, `context`.
*   `forage_observer`: Needs `species`, `location`, `season`.

In [ ]:
def seal_instruction(command: str, params: dict, source: str = "qwen_wendy_ai"):
    """
    Seals an AI instruction into a PSIVI-compliant .psvc + .json sidecar.
    
    Args:
        command: The action type (e.g., 'spawn_agent').
        params: Dictionary of arguments for the command.
        source: Identifier for the requesting agent.
    
    Returns:
        Tuple[str, str]: Paths to the sealed .psvc and .json files.
    """
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Generate unique filename slug
    ts_str = datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S%f')[:-3]
    iso_ts = get_zulu_time_ms()
    filename_base = f"instruction_{command}_{ts_str}"
    
    # 1. Create JSON Sidecar
    sidecar_data = {
        "command": command,
        "params": params,
        "source": source,
        "timestamp": iso_ts,
        "rfc1001_compliant": True
    }
    
    sidecar_path = OUTPUT_DIR / f"{filename_base}.json"
    with open(sidecar_path, 'w') as f:
        json.dump(sidecar_data, f, indent=2)
        
    # 2. Create Binary .psvc Container (Intent Vector)
    # Using deterministic seed based on timestamp for reproducibility if needed
    rng = np.random.default_rng(hash(iso_ts))
    intent_vector = rng.standard_normal(4096).astype(np.float32)
    intent_vector /= np.linalg.norm(intent_vector)
    
    vector_bytes = intent_vector.tobytes()
    compressed = zlib.compress(vector_bytes, level=9)
    
    header = MAGIC
    header += struct.pack('B', VERSION)
    header += struct.pack('B', PRECISION_FLOAT32)
    header += struct.pack('<I', len(intent_vector))
    header += struct.pack('<I', len(compressed))
    
    psvc_path = OUTPUT_DIR / f"{filename_base}.psvc"
    psvc_path.write_bytes(header + compressed)
    
    print(f"✅ Instruction Sealed:")
    print(f"   JSON: {sidecar_path}")
    print(f"   PSVC: {psvc_path}")
    print(f"   Timestamp: {iso_ts}")
    
    return str(psvc_path), str(sidecar_path)

## 🚀 EXAMPLE 1: SPAWN SATELLITE OBSERVER

Scenario: You want to monitor the Goldstream Watershed using RADARSAT SAR data.

In [ ]:
# Define the hypothesis/action
action_params = {
    "template_name": "satellite_observer",
    "agent_name": "goldstream_radarsat_v1",
    "config": {
        "target_region": "Goldstream_Watershed",
        "modality": "RADARSAT_SAR",
        "priority": "HIGH"
    }
}

# Execute Seal
psvc_file, json_file = seal_instruction(
    command="spawn_agent",
    params=action_params,
    source="qwen_wendy_ai"
)

## 🚀 EXAMPLE 2: RESOLVE LITERATURE CONFLICT

Scenario: High fragility detected in gene `IGFBP7`. Spawn a resolver to find ground truth.

In [ ]:
# Define the hypothesis/action
resolver_params = {
    "template_name": "literature_resolver",
    "agent_name": "resolve_igfbp7_osdr",
    "config": {
        "gene": "IGFBP7",
        "organism": "Mus_musculus",
        "context": "Spaceflight_Biology_OSDR"
    }
}

# Execute Seal
psvc_file_res, json_file_res = seal_instruction(
    command="spawn_agent",
    params=resolver_params,
    source="qwen_wendy_ai"
)

## 📡 READ MESH STATE (FEEDBACK LOOP)

After committing the files above to GitHub, wait ~30 seconds, then run this cell to see how the Mesh reacted.

In [ ]:
def read_mesh_state():
    url = "https://raw.githubusercontent.com/psivicom/psivicom.github.io/main/reports/pilot_report.json"
    try:
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        data = resp.json()
        
        print("📊 Current Mesh State:")
        print(f"   Last Sync: {data.get('last_updated')}")
        print(f"   Total Processed: {data.get('total_processed_instructions')}")
        print(f"   Aggregate Fragility: {data.get('aggregate_fragility')}")
        print(f"   Aggregate Concordance: {data.get('aggregate_concordance')}")
        
        # Determine Elasticity Logic
        frag = data.get('aggregate_fragility', 0)
        conc = data.get('aggregate_concordance', 0)
        
        if frag == 0 and conc == 0:
            status = "IDLE"
        elif frag > conc:
            status = "CONTRACT (Resolve conflicts!)"
        elif conc > frag * 2:
            status = "EXPAND (Scale up data collection!)"
        else:
            status = "STABLE"
            
        print(f"   System Elasticity: {status}")
        
        return data
        
    except Exception as e:
        print(f"❌ Error reading mesh state: {e}")
        return None

# Execute Read
state = read_mesh_state()

## 💾 EXPORT FOR GIT COMMIT

Since you are likely running this locally or in a sandbox, use this helper to list the files you need to commit to `psivicom.github.io`.

**Instructions for Wendy:**
1. Take the paths printed in the previous cells.
2. Use your file-writing tool or Git integration to push these two files (`.json` and `.psvc`) to the repository under `data/instruction_queue/`.
3. Trigger the GitHub Action (or wait for the listener).

In [ ]:
files_to_commit = [
    str(json_file),
    str(psvc_file),
    str(json_file_res),
    str(psvc_file_res)
]

print("\n👇 FILES READY FOR GIT COMMIT:")
for f in files_to_commit:
    print(f"   - {f}")

print("\n️ REMINDER: Ensure timestamps are Zulu ms. Ensure RFC 1001 compliance.")